# Extract pressure level from model level data

In [1]:
import numpy as np
import xarray as xr
import os

from geocat.comp import interp_hybrid_to_pressure
from dask.diagnostics import ProgressBar


In [2]:
from dask.distributed import Client
from ncar_jobqueue import NCARCluster

In [3]:
cluster = NCARCluster(project='P93300642',interface='ext',walltime="12:00:00")
cluster
cluster.scale(jobs=16)
client = Client(cluster)
client

/glade/u/apps/opt/conda/envs/npl-2025a/lib/python3.12/site-packages/dask_jobqueue/core.py:266: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/u/apps/opt/conda/envs/npl-2025a/lib/python3.12/site-packages/dask_jobqueue/core.py:285: FutureWarning: env_extra has been renamed to job_script_prologue. You are still using it (even if only set to []; please also check config files). If you did not set job_script_prologue yet, env_extra will be respected for now, but it will be removed in a future release. If you already set job_script_prologue, env_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/u/apps/opt/con

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/rneale/Casper_compute/proxy/37073/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/rneale/Casper_compute/proxy/37073/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.173:35173,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/rneale/Casper_compute/proxy/37073/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [12]:
''' Data Source Settings '''

# 3d->2d varaibles
lev_new = 700.
vars_3d = ['DTCOND']

lens = 2

# CESM3

if lens == 2:
    
    in_dir = '/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm/proc/tseries/month_1/'
    in_case = 'b.e21.BHISTcmip6.f09_g17.LE2-1001.001'
    
    in_case_suff = ['.185001-185912','.186001-186912','.187001-187912','.188001-188912','.189001-189912']





if lens == 2:
    
    in_dir = '/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm/proc/tseries/month_1/'
    in_case = 'b.e21.BHISTcmip6.f09_g17.LE2-1001.001'
    
    in_case_suff = ['.185001-185912','.186001-186912','.187001-187912','.188001-188912','.189001-189912']



## LENS1

if lens == 1:

    in_dir = '/glade/campaign/cesm/collections/cesmLE/CESM-CAM5-BGC-LE/atm/proc/tseries/monthly/'
    in_case = 'b.e11.B1850C5CN.f09_g16.005'
    
    in_case_suff = ['.040001-049912']





for ivar,var in enumerate(vars_3d):

    var_2d = vars_3d[ivar]+str(int(lev_new))
    print('-- ',var_2d,' --')
    
    for itime,ic_suff in enumerate(in_case_suff):
    
            
        
            ic_suff = ic_suff+'.nc'
        
            '''  Run Source Specific Settings '''
            
            in_case_pref = in_case+'.cam.h0.'
            
            
            out_dir = '/glade/work/rneale/python-netcdf/enso/'+var_2d+'/'
            
            
            
            out_file = in_case_pref+var_2d+ic_suff
            
            in_vdir = in_dir+vars_3d[ivar]+'/'
            in_pdir = in_dir+'PS/'
            
            in_vfile = in_case_pref+vars_3d[ivar]+ic_suff
            in_pfile = in_case_pref+'PS'+ic_suff
            
            
            os.makedirs(out_dir, exist_ok=True)
            
            
            p0 = 100000  # surface reference pressure in Pascals
            
            
            
            # Specify output pressure levels
            new_levels = np.array([lev_new])
            new_levels = new_levels * 100  # convert to Pascals
            
            
            
            # Extract the data needed
            
            print('- Read in file...')
            print(in_vdir+in_vfile)
            
            ds_var = xr.open_dataset(in_vdir+in_vfile, chunks={"time": 1, "lat": 192, "lon": 288})
            
        
            ds_var = ds_var.isel(lev=slice(14, 26))
            print(ds_var.lev)
            da_var = ds_var[vars_3d[ivar]]
            
            hyam = ds_var['hyam']  # hybrid A coefficiet
            hybm = ds_var['hybm']  # hybrid B coefficient
            
            ds_ps = xr.open_dataset(in_pdir+in_pfile, chunks={"time": 1, "lat": 192, "lon": 288}) 
            da_ps = ds_ps['PS']
            
            if hyam.ndim == 2: hyam = hyam[0]
            if hybm.ndim == 2: hybm = hybm[0]
            
            
            # Interpolate pressure coordinates form hybrid sigma coord
                
            
            print('- Interpolating')
            da_var = interp_hybrid_to_pressure(da_var,
                                  da_ps,
                                  hyam,
                                  hybm,
                                  p0=p0,
                                  new_levels=new_levels,
                                  method='log')
            # Rename and swap variable name
            da_var = da_var.rename(var_2d).squeeze()
            da_var = da_var.rename({'plev': 'lev'})
            
            # Rescale to mb
            da_var = da_var.assign_coords(lev=0.01*da_var.lev)

            print('- Unlazying...')
            with ProgressBar():
                da_out = da_var.compute()  # Turns into in-memory NumPy-backed array

        
            print('- Writing out...')
            da_out.to_netcdf(out_dir+out_file,mode="w")
            
            print('-Done')

--  DTCOND700  --
- Read in file...
/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm/proc/tseries/month_1/DTCOND/b.e21.BHISTcmip6.f09_g17.LE2-1001.001.cam.h0.DTCOND.185001-185912.nc
<xarray.DataArray 'lev' (lev: 12)> Size: 96B
array([197.908087, 232.828619, 273.910817, 322.241902, 379.100904, 445.992574,
       524.687175, 609.778695, 691.38943 , 763.404481, 820.858369, 859.534767])
Coordinates:
  * lev      (lev) float64 96B 197.9 232.8 273.9 322.2 ... 763.4 820.9 859.5
Attributes:
    long_name:      hybrid level at midpoints (1000*(A+B))
    units:          hPa
    positive:       down
    standard_name:  atmosphere_hybrid_sigma_pressure_coordinate
    formula_terms:  a: hyam b: hybm p0: P0 ps: PS


/glade/derecho/scratch/rneale/tmp/ipykernel_124388/669751385.py:91: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 1. This could degrade performance. Instead, consider rechunking after loading.
  ds_var = xr.open_dataset(in_vdir+in_vfile, chunks={"time": 1, "lat": 192, "lon": 288})
/glade/derecho/scratch/rneale/tmp/ipykernel_124388/669751385.py:101: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 1. This could degrade performance. Instead, consider rechunking after loading.
  ds_ps = xr.open_dataset(in_pdir+in_pfile, chunks={"time": 1, "lat": 192, "lon": 288})


- Interpolating
- Unlazying...
- Writing out...
-Done
- Read in file...
/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm/proc/tseries/month_1/DTCOND/b.e21.BHISTcmip6.f09_g17.LE2-1001.001.cam.h0.DTCOND.186001-186912.nc
<xarray.DataArray 'lev' (lev: 12)> Size: 96B
array([197.908087, 232.828619, 273.910817, 322.241902, 379.100904, 445.992574,
       524.687175, 609.778695, 691.38943 , 763.404481, 820.858369, 859.534767])
Coordinates:
  * lev      (lev) float64 96B 197.9 232.8 273.9 322.2 ... 763.4 820.9 859.5
Attributes:
    long_name:      hybrid level at midpoints (1000*(A+B))
    units:          hPa
    positive:       down
    standard_name:  atmosphere_hybrid_sigma_pressure_coordinate
    formula_terms:  a: hyam b: hybm p0: P0 ps: PS


/glade/derecho/scratch/rneale/tmp/ipykernel_124388/669751385.py:91: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 1. This could degrade performance. Instead, consider rechunking after loading.
  ds_var = xr.open_dataset(in_vdir+in_vfile, chunks={"time": 1, "lat": 192, "lon": 288})
/glade/derecho/scratch/rneale/tmp/ipykernel_124388/669751385.py:101: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 1. This could degrade performance. Instead, consider rechunking after loading.
  ds_ps = xr.open_dataset(in_pdir+in_pfile, chunks={"time": 1, "lat": 192, "lon": 288})


- Interpolating
- Unlazying...
- Writing out...
-Done
- Read in file...
/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm/proc/tseries/month_1/DTCOND/b.e21.BHISTcmip6.f09_g17.LE2-1001.001.cam.h0.DTCOND.187001-187912.nc
<xarray.DataArray 'lev' (lev: 12)> Size: 96B
array([197.908087, 232.828619, 273.910817, 322.241902, 379.100904, 445.992574,
       524.687175, 609.778695, 691.38943 , 763.404481, 820.858369, 859.534767])
Coordinates:
  * lev      (lev) float64 96B 197.9 232.8 273.9 322.2 ... 763.4 820.9 859.5
Attributes:
    long_name:      hybrid level at midpoints (1000*(A+B))
    units:          hPa
    positive:       down
    standard_name:  atmosphere_hybrid_sigma_pressure_coordinate
    formula_terms:  a: hyam b: hybm p0: P0 ps: PS
- Interpolating


...

<xarray.DataArray 'lev' (lev: 12)> Size: 96B
array([197.908087, 232.828619, 273.910817, 322.241902, 379.100904, 445.992574,
       524.687175, 609.778695, 691.38943 , 763.404481, 820.858369, 859.534767])
Coordinates:
  * lev      (lev) float64 96B 197.9 232.8 273.9 322.2 ... 763.4 820.9 859.5
Attributes:
    long_name:      hybrid level at midpoints (1000*(A+B))
    units:          hPa
    positive:       down
    standard_name:  atmosphere_hybrid_sigma_pressure_coordinate
    formula_terms:  a: hyam b: hybm p0: P0 ps: PS
- Interpolating
- Unlazying...


/glade/derecho/scratch/rneale/tmp/ipykernel_124388/669751385.py:101: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 1. This could degrade performance. Instead, consider rechunking after loading.
  ds_ps = xr.open_dataset(in_pdir+in_pfile, chunks={"time": 1, "lat": 192, "lon": 288})


- Writing out...
-Done
- Read in file...
/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm/proc/tseries/month_1/DTCOND/b.e21.BHISTcmip6.f09_g17.LE2-1001.001.cam.h0.DTCOND.189001-189912.nc


/glade/derecho/scratch/rneale/tmp/ipykernel_124388/669751385.py:91: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 1. This could degrade performance. Instead, consider rechunking after loading.
  ds_var = xr.open_dataset(in_vdir+in_vfile, chunks={"time": 1, "lat": 192, "lon": 288})
/glade/derecho/scratch/rneale/tmp/ipykernel_124388/669751385.py:101: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 1. This could degrade performance. Instead, consider rechunking after loading.
  ds_ps = xr.open_dataset(in_pdir+in_pfile, chunks={"time": 1, "lat": 192, "lon": 288})


<xarray.DataArray 'lev' (lev: 12)> Size: 96B
array([197.908087, 232.828619, 273.910817, 322.241902, 379.100904, 445.992574,
       524.687175, 609.778695, 691.38943 , 763.404481, 820.858369, 859.534767])
Coordinates:
  * lev      (lev) float64 96B 197.9 232.8 273.9 322.2 ... 763.4 820.9 859.5
Attributes:
    long_name:      hybrid level at midpoints (1000*(A+B))
    units:          hPa
    positive:       down
    standard_name:  atmosphere_hybrid_sigma_pressure_coordinate
    formula_terms:  a: hyam b: hybm p0: P0 ps: PS
- Interpolating
- Unlazying...
- Writing out...
-Done
